# Plaxis 2D Shivam

______________________________________________________________________

**Authors: Pablo Vasconez**

#### 1. Install additional requirements and import required modules

In [ ]:
%pip install openpyxl

In [ ]:
import pandas as pd
from plxcontroller.plaxis_2d_input_controller import Plaxis2DInputController
from plxcontroller.plaxis_2d_output_controller import Plaxis2DOutputController
from plxcontroller.precalculation_point_2d import PrecalculationPoint2D
from plxcontroller.parameter_optimization_2d import ParameterOptimization2D, MaterialParameterInput, DependentParameter, MeasuredValueAtPoint

#### 2. Define connection settings 

In [ ]:
# Enter IP address of machine and port (integer) and password of the PLAXIS remote server
ip_address = "localhost"
input_port = 10000
output_port = 10001

#### 3. Define model path

In [ ]:
# Define model path
model_path = r"original_model.p2dx"

#### 4. Build parameter optimization object

In [ ]:
# Initialize the parameter optimization 
po = ParameterOptimization2D(
    model_path=model_path,
    ip_address=ip_address,
    input_port=input_port,
    output_port=output_port,
)

In [ ]:
# Parse material parameter inputs from CSV file
material_parameter_inputs_df = pd.read_csv(r"material_parameter_inputs.csv")
display(material_parameter_inputs_df)

po.material_parameter_inputs = []
for _, row in material_parameter_inputs_df.iterrows():
    po.material_parameter_inputs.append(
        MaterialParameterInput(
            material_identification=row["material_identification"],
            parameter_name=row["parameter_name"],
            initial_value=row["initial_value"],
            min_value=row["min_value"],
            max_value=row["max_value"],
            dependent_parameters=[
                DependentParameter(name=row["dependent_parameter"], ratio=row["dependency_ratio"])
                if not pd.isna(row["dependent_parameter"])
                else None
            ]
            ,
        )
    )

In [ ]:
# Parse measured data points from CSV file
measured_data_points_df = pd.read_csv(r"measured_data_original.csv", delimiter=";")

# Remove data at time 0
measured_data_points_df = measured_data_points_df[measured_data_points_df["time"] > 0]

# Remove data with point_identification starting with "IC" and result_type "Soil.Ux"
measured_data_points_df = measured_data_points_df[~((measured_data_points_df["point_identification"].str.startswith("IC")) & (measured_data_points_df["result_type"] == "Soil.Ux"))]

# Change all data with point_identification starting with "IC" and result_type "Soil.Uy" to have result_type "Soil.Ux"
measured_data_points_df.loc[(measured_data_points_df["point_identification"].str.startswith("IC")) & (measured_data_points_df["result_type"] == "Soil.Uy"), "result_type"] = "Soil.Ux"

# Add a column "relative_to_point" with empty values
measured_data_points_df["relative_to_point"] = ""

# For all data with point_identification starting with "IC", set the value of "relative_to_point" to be the same as the point_identification but replacing everything after "_" by "_-11,8"
measured_data_points_df.loc[measured_data_points_df["point_identification"].str.startswith("IC"), "relative_to_point"] = measured_data_points_df["point_identification"].str.replace(r"_(.*)", "_-11,8", regex=True)

# Remove all data with "IC" and "-11,8" in the point_identification
measured_data_points_df = measured_data_points_df[~((measured_data_points_df["point_identification"].str.startswith("IC")) & (measured_data_points_df["point_identification"].str.contains("-11,8")))]

# Remove all data with "IC" and "surface" in the point_identification
measured_data_points_df = measured_data_points_df[~((measured_data_points_df["point_identification"].str.startswith("IC")) & (measured_data_points_df["point_identification"].str.contains("surface")))]

# Save to CSV file
measured_data_points_df.to_csv(r"measured_data.csv", index=False, sep=";")
measured_data_points_df

In [ ]:
# Parse measured data points from CSV file
measured_data_points_df = pd.read_csv(r"measured_data.csv", delimiter=";")
display(measured_data_points_df)

# Compute the number of measured values with "Soil.Ux" results
num_soil_ux_values = len(measured_data_points_df[measured_data_points_df["result_type"] == "Soil.Ux"])
print(f"Number of measured values with 'Soil.Ux' results: {num_soil_ux_values}")

# Compute the number of measured values with "Soil.Uy" results
num_soil_uy_values = len(measured_data_points_df[measured_data_points_df["result_type"] == "Soil.Uy"])
print(f"Number of measured values with 'Soil.Uy' results: {num_soil_uy_values}")

# Compute the weight of "Soil.Uy" and "Soil.Ux" results
weight_uy = 1.0
weight_ux = num_soil_uy_values / num_soil_ux_values
print(f"Weight of 'Soil.Uy' results: {weight_uy}")
print(f"Weight of 'Soil.Ux' results: {weight_ux}")

po.measured_values = []
for _, row in measured_data_points_df.iterrows():
    po.measured_values.append(
        MeasuredValueAtPoint(
            point_identification=row["point_identification"],
            result_type=row["result_type"],
            time=row["time"],
            value=row["value"],
            weight=weight_uy if row["result_type"] == "Soil.Uy" else weight_ux,
            relative_to_point=row["relative_to_point"] if row["relative_to_point"] != "" else None,
        )
    )

#### 4. Select points and recalculate model

In [ ]:
# Create a new Plaxis controller instance
ci = Plaxis2DInputController()
# Connect to the Plaxis remote server.
# Note: This function expects that you have the following environmental variables set:
# "PLAXIS_2D_INPUT_PROGRAM", "PLAXIS_2D_PASSWORD"
ci.connect(ip_address, input_port)
ci.open(model_path)

In [ ]:
# Load precalculation points from excel file
df = pd.read_excel(r"Node coördinaten.xlsx")
df

In [ ]:
# Parse the dataframe as a list of PrecalculationPoint2D
points_to_select = []
for i, row in df.iterrows():
    points_to_select.append(
        PrecalculationPoint2D(point_type="node", x=row["x"], y=row["y"], identification=row["Node name"])
    )

In [ ]:
# Select precalculation points
ci.select_precalculation_points(
    points=points_to_select, delete_previously_selected=True
)

In [ ]:
# Recalculate the whole model and kill the subprocess
ci.recalculate_all_phases()
ci.kill()

#### 6. Get results

In [ ]:
# Create connection with Plaxis output and open model
co = Plaxis2DOutputController()
co.connect(ip_address, output_port)
co.open(model_path)

In [ ]:
# Request node time history results
phase_numbers = [co.get_phase_number_from_phase_name(phase) for phase in co.g_o.Phases[1:]]
result_type_names=["Soil.Ux", "Soil.Uy"]

results = co.get_node_time_history_results(phase_numbers=phase_numbers, result_type_names=result_type_names)

In [ ]:
# Show results as dataframe
df = results.to_dataframe()
df

In [ ]:
# Close and kill output program
co.close()
co.kill()